# Industrial Energy Forecasting — Data Preparation
**Member 1 contribution | Syed Fahim Hussain**

This notebook audits the original UCI CSV and exports a checked, chronologically sorted copy. It does not train models, choose split dates, impute values, scale features or remove high-consumption observations. Original measurements and source column names are preserved.

Source: UCI Steel Industry Energy Consumption, https://doi.org/10.24432/C52G8C (CC BY 4.0). Original file supplied by the group. Results below are calculated from that file, not copied from the proposal.

Run all cells in order. Generated files are overwritten on rerun; the original CSV is never overwritten.

## 1 Locate the project and import packages
The relative layout works when Jupyter starts in the project root or in `all_experiments`. No personal Windows path is embedded.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import pandas as pd
import numpy as np

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'all_datasets' / 'Steel_industry_data.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Extract the entire project ZIP and open its notebook.')
SOURCE = ROOT / 'all_datasets' / 'Steel_industry_data.csv'
OUT = ROOT / 'outputs' / 'data_quality'
OUT.mkdir(parents=True, exist_ok=True)
(ROOT / 'reports').mkdir(exist_ok=True)
source_hash = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
print('Python:', platform.python_version(), '| pandas:', pd.__version__, '| NumPy:', np.__version__)
print('Source SHA256:', source_hash)

Python: 3.12.14 | pandas: 3.0.6 | NumPy: 2.5.3
Source SHA256: 9b1cee6f9cb9cd9df2b95814ca90a9a2ff15b7f5f1fba0fae3c643e82072eacc


## 2 Load and inspect the source
There should be 11 source columns. `Usage_kWh` is the future forecasting target, but its historical observations may later become lagged predictors. Loading numeric types is not feature scaling.

In [2]:
raw = pd.read_csv(SOURCE)
expected = ['date', 'Usage_kWh', 'Lagging_Current_Reactive.Power_kVarh',
            'Leading_Current_Reactive_Power_kVarh', 'CO2(tCO2)',
            'Lagging_Current_Power_Factor', 'Leading_Current_Power_Factor',
            'NSM', 'WeekStatus', 'Day_of_week', 'Load_Type']
assert list(raw.columns) == expected, 'Unexpected schema: review before proceeding.'
print('Rows and columns:', raw.shape)
print(raw.head(3).to_string(index=False))
print('\nData types:\n', raw.dtypes.to_string())

Rows and columns: (35040, 11)
            date  Usage_kWh  Lagging_Current_Reactive.Power_kVarh  Leading_Current_Reactive_Power_kVarh  CO2(tCO2)  Lagging_Current_Power_Factor  Leading_Current_Power_Factor  NSM WeekStatus Day_of_week  Load_Type
01/01/2018 00:15       3.17                                  2.95                                   0.0        0.0                         73.21                         100.0  900    Weekday      Monday Light_Load
01/01/2018 00:30       4.00                                  4.46                                   0.0        0.0                         66.77                         100.0 1800    Weekday      Monday Light_Load
01/01/2018 00:45       3.24                                  3.28                                   0.0        0.0                         70.28                         100.0 2700    Weekday      Monday Light_Load

Data types:
 date                                        str
Usage_kWh                               float64
Lagg

## 3 Check missing values and duplicate records
Blank strings are checked separately from missing cells. Duplicate timestamps and duplicated measurements are different: identical measurements at different times can be genuine and should not be removed.

In [3]:
missing = raw.isna().sum()
blank = pd.Series({c: int(raw[c].astype(str).str.strip().eq('').sum()) for c in raw})
missing_table = pd.DataFrame({'missing_cells': missing, 'blank_strings': blank})
missing_table.to_csv(OUT / 'missing_values.csv', index_label='column')
duplicate_rows = int(raw.duplicated().sum())
print(missing_table.to_string())
print('Exact duplicate rows:', duplicate_rows)
assert missing.sum() == 0 and blank.sum() == 0, 'Review missing data before export.'
assert duplicate_rows == 0, 'Review duplicate records before export.'

                                      missing_cells  blank_strings
date                                              0              0
Usage_kWh                                         0              0
Lagging_Current_Reactive.Power_kVarh              0              0
Leading_Current_Reactive_Power_kVarh              0              0
CO2(tCO2)                                         0              0
Lagging_Current_Power_Factor                      0              0
Leading_Current_Power_Factor                      0              0
NSM                                               0              0
WeekStatus                                        0              0
Day_of_week                                       0              0
Load_Type                                         0              0
Exact duplicate rows: 0


## 4 Parse dates explicitly and inspect the original order
Use day/month/year (`%d/%m/%Y %H:%M`), not an ambiguous automatic date parser. Preserve the original row number in the audit so every exported row can be traced back to the source.

In [4]:
parsed = pd.to_datetime(raw['date'], format='%d/%m/%Y %H:%M', errors='coerce')
assert parsed.notna().all(), 'Unparseable timestamps require review.'
duplicate_timestamps = int(parsed.duplicated().sum())
assert duplicate_timestamps == 0, 'Duplicate timestamps require review.'
raw_deltas = parsed.diff()
delta_table = raw_deltas.dropna().astype(str).value_counts().rename_axis('original_row_delta').reset_index(name='count')
delta_table.to_csv(OUT / 'original_timestamp_deltas.csv', index=False)
print('Recorded date range:', parsed.min(), 'to', parsed.max())
print('Already chronological:', parsed.is_monotonic_increasing)
print('Duplicate timestamps:', duplicate_timestamps)
print(delta_table.to_string(index=False))
midnight = parsed.dt.hour.eq(0) & parsed.dt.minute.eq(0)
print('Midnight rows:', int(midnight.sum()))
print(raw.loc[midnight, ['date','NSM','Day_of_week']].head().to_string())

Recorded date range: 2018-01-01 00:00:00 to 2018-12-31 23:45:00
Already chronological: False
Duplicate timestamps: 0
original_row_delta  count
   0 days 00:15:00  34310
 -1 days +00:15:00    365
   1 days 00:15:00    364
Midnight rows: 365
                 date  NSM Day_of_week
95   01/01/2018 00:00    0      Monday
191  02/01/2018 00:00    0     Tuesday
287  03/01/2018 00:00    0   Wednesday
383  04/01/2018 00:00    0    Thursday
479  05/01/2018 00:00    0      Friday


**Adopted interpretation (updated 19 October 2026):** The original file places each day's 00:00 record after its 23:45 record. The team interprets midnight as the following day's interval endpoint and adds one day to those timestamps. This preserves source measurement order and produces a continuous 15-minute sequence. This is a documented assumption based on row order, not independent confirmation of the author's intention. All calendar fields describe the adopted endpoint timestamp; raw calendar labels remain available in the original CSV and the change audit. Load_Type is preserved, not regenerated or assumed to be known in advance.

## 5 Apply the midnight interpretation and verify the time grid
Start from the original CSV on every run so midnight dates are shifted only once. Check the source pattern before modifying dates. Keep a detailed timestamp and calendar change audit.

In [5]:
# Guard the assumption: every midnight row follows 23:45 on its recorded date.
previous = parsed.shift(1)
assert (previous.loc[midnight] == parsed.loc[midnight] + pd.Timedelta(hours=23, minutes=45)).all()
assert int(midnight.sum()) == 365
adjusted = parsed + pd.to_timedelta(midnight.astype(int), unit='D')
assert adjusted.is_monotonic_increasing
assert adjusted.diff().dropna().eq(pd.Timedelta(minutes=15)).all()
assert not adjusted.duplicated().any()

work = raw.copy()
work['date'] = adjusted
work['Day_of_week'] = adjusted.dt.day_name()
work['WeekStatus'] = np.where(adjusted.dt.dayofweek >= 5, 'Weekend', 'Weekday')
work['NSM'] = (adjusted.dt.hour * 3600 + adjusted.dt.minute * 60).astype(raw['NSM'].dtype)
work['_source_row'] = np.arange(len(raw))
work = work.sort_values('date', kind='stable').reset_index(drop=True)
checked = work.drop(columns='_source_row')
expected_grid = pd.date_range(checked.date.min(), checked.date.max(), freq='15min')
missing_times = expected_grid.difference(pd.DatetimeIndex(checked.date))
off_grid = checked.loc[~checked.date.isin(expected_grid), ['date']]
pd.DataFrame({'missing_timestamp': missing_times}).to_csv(OUT / 'missing_timestamps.csv', index=False)
off_grid.to_csv(OUT / 'off_grid_timestamps.csv', index=False)
assert len(missing_times) == 0 and len(off_grid) == 0
assert checked.date.diff().dropna().eq(pd.Timedelta(minutes=15)).all()
assert checked.shape == raw.shape == (35040, 11)
assert checked.date.iloc[0] == pd.Timestamp('2018-01-01 00:15')
assert checked.date.iloc[-1] == pd.Timestamp('2019-01-01 00:00')
lineage = pd.DataFrame({'checked_row': np.arange(len(work)),
                        'source_row': work['_source_row'],
                        'original_timestamp': parsed.iloc[work['_source_row']].to_numpy(),
                        'adopted_timestamp': checked.date,
                        'midnight_shifted': midnight.iloc[work['_source_row']].to_numpy()})
lineage.to_csv(OUT / 'row_lineage.csv', index=False)
changes = pd.DataFrame({'source_row': np.arange(len(raw)), 'original_timestamp': parsed,
                        'adopted_timestamp': adjusted})
for c in ['Day_of_week', 'WeekStatus', 'NSM']:
    changes['original_' + c] = raw[c]
    changes['adopted_' + c] = work.sort_values('_source_row')[c].to_numpy()
changes.loc[midnight].to_csv(OUT / 'midnight_adjustments.csv', index=False)
calendar_changes = {c: int(raw[c].ne(work.sort_values('_source_row')[c].reset_index(drop=True)).sum())
                    for c in ['Day_of_week', 'WeekStatus', 'NSM']}
print('Midnight timestamps shifted:', int(midnight.sum()))
print('Calendar values changed:', calendar_changes)
print('Adopted range:', checked.date.min(), 'to', checked.date.max())
print('Missing timestamps:', len(missing_times), '| Off-grid timestamps:', len(off_grid))
print('Rows changing position relative to source:', int(lineage.checked_row.ne(lineage.source_row).sum()))


Midnight timestamps shifted: 365
Calendar values changed: {'Day_of_week': 365, 'WeekStatus': 104, 'NSM': 0}
Adopted range: 2018-01-01 00:15:00 to 2019-01-01 00:00:00
Missing timestamps: 0 | Off-grid timestamps: 0
Rows changing position relative to source: 0


## 6 Verify calendar labels and category values
Calendar fields are recalculated from the adopted interval endpoint timestamps. NSM is seconds since midnight; weekdays are Monday–Friday. These checks establish internal consistency, not confirmation of the timestamp assumption.

In [6]:
calendar_checks = {
    'Day_of_week_mismatches': int(checked.Day_of_week.ne(checked.date.dt.day_name()).sum()),
    'WeekStatus_mismatches': int(checked.WeekStatus.ne(np.where(checked.date.dt.dayofweek >= 5, 'Weekend', 'Weekday')).sum()),
    'NSM_mismatches': int(checked.NSM.ne(checked.date.dt.hour*3600 + checked.date.dt.minute*60).sum())
}
print(calendar_checks)
assert all(v == 0 for v in calendar_checks.values()), 'Calendar inconsistency requires review.'
for col, allowed in {'WeekStatus': {'Weekday','Weekend'},
                     'Day_of_week': {'Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'},
                     'Load_Type': {'Light_Load','Medium_Load','Maximum_Load'}}.items():
    values = set(checked[col].unique())
    assert values <= allowed, f'Unexpected category in {col}'
    print(col, ':', sorted(values))

{'Day_of_week_mismatches': 0, 'WeekStatus_mismatches': 0, 'NSM_mismatches': 0}
WeekStatus : ['Weekday', 'Weekend']
Day_of_week : ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']
Load_Type : ['Light_Load', 'Maximum_Load', 'Medium_Load']


## 7 Check numeric validity without removing genuine peaks
Check finite values, nonnegative energy/reactive readings, power factors within 0–100 and NSM within a day. Zero consumption may represent inactivity; it is retained. Minimum/maximum values below are quality diagnostics, not thresholds for feature selection. Distribution-driven analysis and outlier thresholds should be developed on training data by the relevant members.

In [7]:
numeric = checked.select_dtypes(include='number')
ranges = pd.DataFrame({'minimum': numeric.min(), 'maximum': numeric.max(),
                       'zero_count': numeric.eq(0).sum(),
                       'negative_count': numeric.lt(0).sum(),
                       'nonfinite_count': (~np.isfinite(numeric)).sum()})
ranges.to_csv(OUT / 'numeric_range_checks.csv', index_label='column')
print(ranges.to_string())
assert np.isfinite(numeric.to_numpy()).all()
assert numeric.ge(0).all().all(), 'Negative readings require review.'
for col in ['Lagging_Current_Power_Factor','Leading_Current_Power_Factor']:
    assert checked[col].between(0,100).all()
assert checked.NSM.between(0,86399).all()
assert (checked.NSM % 900 == 0).all()
print('No rows removed, capped, imputed or scaled.')

                                      minimum   maximum  zero_count  negative_count  nonfinite_count
Usage_kWh                                 0.0    157.18           1               0                0
Lagging_Current_Reactive.Power_kVarh      0.0     96.91        7194               0                0
Leading_Current_Reactive_Power_kVarh      0.0     27.76       23610               0                0
CO2(tCO2)                                 0.0      0.07       20990               0                0
Lagging_Current_Power_Factor              0.0    100.00           1               0                0
Leading_Current_Power_Factor              0.0    100.00           1               0                0
NSM                                       0.0  85500.00         365               0                0
No rows removed, capped, imputed or scaled.


## 8 Export and independently verify preservation
The output retains 11 columns. Timestamp dates and derived calendar labels may change under the adopted convention. All other fields, including every measurement and Load_Type, must exactly match the source after export and reload. The original CSV hash must remain unchanged.

In [8]:
DEST = ROOT / 'all_datasets' / 'steel_checked.csv'
checked.to_csv(DEST, index=False, date_format='%Y-%m-%d %H:%M:%S')
reloaded = pd.read_csv(DEST)
reloaded['date'] = pd.to_datetime(reloaded['date'], format='%Y-%m-%d %H:%M:%S')
pd.testing.assert_frame_equal(checked, reloaded, check_exact=True)
original_order = work.sort_values('_source_row').drop(columns='_source_row').reset_index(drop=True)
expected_original = raw.copy()
expected_original['date'] = adjusted
expected_original['Day_of_week'] = adjusted.dt.day_name()
expected_original['WeekStatus'] = np.where(adjusted.dt.dayofweek >= 5, 'Weekend', 'Weekday')
expected_original['NSM'] = (adjusted.dt.hour * 3600 + adjusted.dt.minute * 60).astype(raw['NSM'].dtype)
measurement_cols = [c for c in raw if c not in ['date', 'Day_of_week', 'WeekStatus', 'NSM']]
pd.testing.assert_frame_equal(original_order[measurement_cols], raw[measurement_cols], check_exact=True)
pd.testing.assert_frame_equal(original_order, expected_original, check_exact=True)
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == source_hash
summary = {
 'rows': len(raw), 'columns': len(raw.columns), 'missing_cells': int(missing.sum()),
 'exact_duplicate_rows': duplicate_rows, 'duplicate_timestamps': duplicate_timestamps,
 'first_recorded_timestamp': str(parsed.min()), 'last_recorded_timestamp': str(parsed.max()),
 'first_adopted_timestamp': str(checked.date.min()), 'last_adopted_timestamp': str(checked.date.max()),
 'midnight_timestamps_shifted': int(midnight.sum()), 'calendar_values_changed': calendar_changes,
 'adopted_duplicate_timestamps': int(checked.date.duplicated().sum()),
 'missing_15min_timestamps_within_bounds': len(missing_times),
 'original_backward_timestamp_steps': int(raw_deltas.lt(pd.Timedelta(0)).sum()),
 'midnight_rows': int(midnight.sum()), 'rows_removed': 0, 'measurements_changed': 0,
 'calendar_checks': calendar_checks, 'timestamp_policy': 'Shift original midnight dates forward one day; derive calendar fields from adopted endpoint timestamps; preserve measurement order.',
 'timestamp_limitation': 'Next-day midnight is a team assumption based on source row order, not independently confirmed author intent.',
 'source_sha256': source_hash,
 'checked_sha256': hashlib.sha256(DEST.read_bytes()).hexdigest(),
 'environment': {'python':platform.python_version(), 'pandas':pd.__version__, 'numpy':np.__version__}
}
(OUT / 'quality_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print('Export round trip and source preservation checks passed.')

{
  "rows": 35040,
  "columns": 11,
  "missing_cells": 0,
  "exact_duplicate_rows": 0,
  "duplicate_timestamps": 0,
  "first_recorded_timestamp": "2018-01-01 00:00:00",
  "last_recorded_timestamp": "2018-12-31 23:45:00",
  "first_adopted_timestamp": "2018-01-01 00:15:00",
  "last_adopted_timestamp": "2019-01-01 00:00:00",
  "midnight_timestamps_shifted": 365,
  "calendar_values_changed": {
    "Day_of_week": 365,
    "WeekStatus": 104,
    "NSM": 0
  },
  "adopted_duplicate_timestamps": 0,
  "missing_15min_timestamps_within_bounds": 0,
  "original_backward_timestamp_steps": 365,
  "midnight_rows": 365,
  "rows_removed": 0,
  "measurements_changed": 0,
  "calendar_checks": {
    "Day_of_week_mismatches": 0,
    "WeekStatus_mismatches": 0,
    "NSM_mismatches": 0
  },
  "timestamp_policy": "Shift original midnight dates forward one day; derive calendar fields from adopted endpoint timestamps; preserve measurement order.",
  "timestamp_limitation": "Next-day midnight is a team assumption 

## 9 Write the report contribution and handover
The generated text describes this completed audit only. Other members should add their own feature engineering, modelling and evaluation results. The original calendar and electrical columns are retained for traceability; retaining a column does not approve it as a predictor.

In [9]:
report = f"""# Data Description and Data Preprocessing — Fahim

## Dataset and audit
The UCI Steel Industry Energy Consumption dataset (https://doi.org/10.24432/C52G8C) contains {len(raw):,} rows and {len(raw.columns)} columns. Usage_kWh records electricity consumption. The audit found no missing cells, blank strings, exact duplicate rows or duplicate timestamps. Repeated measurements at different times are not automatically treated as duplicate records. Numeric values were finite and nonnegative; power factors were between 0 and 100. These checks do not establish that every sensor reading is accurate.

## Adopted timestamp interpretation
In the original row order, each recorded date's 00:00 row follows its 23:45 row. The team interprets these records as the following day's interval endpoint. Accordingly, {int(midnight.sum())} midnight timestamps were shifted forward by one day. This is an assumption based on source ordering, not independent confirmation of the author's intention. It supersedes our preliminary approach of sorting literal timestamps. The resulting series runs from {checked.date.min()} to {checked.date.max()} in uninterrupted 15-minute steps, preserving the original measurement order. Source clarification or a sensitivity analysis could assess this remaining semantic uncertainty.

## Calendar consistency and preservation
Day_of_week, WeekStatus and NSM were recalculated from adopted endpoint timestamps. Changes: {calendar_changes}. This is an endpoint-based calendar convention, including at midnight; original labels remain in the raw file and audit. Load_Type was retained as supplied, without assuming that it is a known production schedule. All {len(raw):,} rows and original measurements were retained. No imputation, capping, scaling, encoding or statistical outlier removal was performed. The unchanged raw CSV, source hash, row lineage and midnight change audit preserve traceability. Export/reload checks confirmed exact preservation of all non-calendar measurement and label columns.

## Handover
Use the updated steel_checked.csv and parse date as ISO datetime. Kapiln will define target-aligned chronological splits, historical features and benchmarks. Kennard should use the agreed training period for exploratory decisions. Daniel will train and select models using training/validation data; Feng will evaluate frozen choices on test data. Calendar inputs for the future target interval are known in advance. Electrical measurements must be historical only. CO2(tCO2) should be excluded from predictors; Load_Type needs an explicit availability decision before use. Fit any learned preprocessing only on training data. Forecasting performance does not demonstrate actual energy savings.
"""
(ROOT / 'reports' / 'member1_report_sections.md').write_text(report, encoding='utf-8')
print(report)


# Data Description and Data Preprocessing — Fahim

## Dataset and audit
The UCI Steel Industry Energy Consumption dataset (https://doi.org/10.24432/C52G8C) contains 35,040 rows and 11 columns. Usage_kWh records electricity consumption. The audit found no missing cells, blank strings, exact duplicate rows or duplicate timestamps. Repeated measurements at different times are not automatically treated as duplicate records. Numeric values were finite and nonnegative; power factors were between 0 and 100. These checks do not establish that every sensor reading is accurate.

## Adopted timestamp interpretation
In the original row order, each recorded date's 00:00 row follows its 23:45 row. The team interprets these records as the following day's interval endpoint. Accordingly, 365 midnight timestamps were shifted forward by one day. This is an assumption based on source ordering, not independent confirmation of the author's intention. It supersedes our preliminary approach of sorting literal

## Completion checklist
- Raw CSV preserved byte-for-byte; all 35,040 measurements retained.
- 365 original midnight timestamps shifted once, according to the documented team assumption.
- Calendar fields recalculated; original and adopted values retained in the audit.
- Continuous 15-minute grid and no duplicate timestamps verified.
- CSV export/reload and measurement preservation checks passed.
